Part A: Load Silver and add local time (cell 1)

In [0]:
%python
from pyspark.sql import functions as F

S = spark.table("workspace.energy.silver_generation")

S = (S.withColumn("ts_berlin", F.from_utc_timestamp("ts_utc", "Europe/Berlin"))
      .withColumn("date", F.to_date("ts_berlin"))
      .withColumn("month", F.date_trunc("month", "ts_berlin").cast("date"))
      .withColumn("year", F.year("ts_berlin"))
      .withColumn("hour", F.hour("ts_berlin")))

# pumped storage is not primary generation, so exclude it from shares
gen = S.filter("category != 'storage'")

display(gen.select("ts_utc", "ts_berlin", "date", "month", "year", "hour").limit(5))

ts_utc,ts_berlin,date,month,year,hour
2020-10-01T00:00:00.000Z,2020-10-01T02:00:00.000Z,2020-10-01,2020-10-01,2020,2
2020-10-01T04:00:00.000Z,2020-10-01T06:00:00.000Z,2020-10-01,2020-10-01,2020,6
2020-10-03T11:00:00.000Z,2020-10-03T13:00:00.000Z,2020-10-03,2020-10-01,2020,13
2020-10-04T08:00:00.000Z,2020-10-04T10:00:00.000Z,2020-10-04,2020-10-01,2020,10
2020-10-05T00:00:00.000Z,2020-10-05T02:00:00.000Z,2020-10-05,2020-10-01,2020,2


Part B: Monthly renewable share (cell 2)

Question answered: how much of German generation came from renewables each month?

In [0]:
%python
monthly = (gen.groupBy("month")
    .agg(F.sum(F.when(F.col("category") == "renewable", F.col("mwh"))).alias("renewable_mwh"),
         F.sum("mwh").alias("total_mwh"))
    .withColumn("renewable_share_pct",
                F.round(F.col("renewable_mwh") / F.col("total_mwh") * 100, 2))
    .orderBy("month"))

monthly.write.mode("overwrite").saveAsTable("workspace.energy.gold_monthly_renewable_share")
display(monthly.orderBy(F.desc("month")).limit(12))

month,renewable_mwh,total_mwh,renewable_share_pct
2026-10-01,1.6097915479999905E7,2.7206930629999936E7,59.17
2026-09-01,2.374763130000034E7,3.667747403000034E7,64.75
2026-08-01,2.4206674159999773E7,3.521168266999984E7,68.75
2026-07-01,2.589852996000002E7,3.6543757970000066E7,70.87
2026-06-01,2.3410045690000042E7,3.3266100120000046E7,70.37
2026-05-01,2.2732115830000013E7,3.3518478840000097E7,67.82
2026-04-01,2.3852301339999948E7,3.5367379599999994E7,67.44
2026-03-01,2.3004461880000018E7,3.877243419000014E7,59.33
2026-02-01,2.1296608579999987E7,4.037972350000002E7,52.74
2026-01-01,2.2592163519999947E7,4.504207372999999E7,50.16


Yearly generation mix (cell 3)

Question answered: how much did each energy source produce per year?

In [0]:
%python
yearly_mix = (gen.groupBy("year", "energy_source", "category")
    .agg(F.round(F.sum("mwh") / 1000, 1).alias("gwh"))
    .orderBy("year", F.desc("gwh")))

yearly_mix.write.mode("overwrite").saveAsTable("workspace.energy.gold_yearly_generation_mix")
display(yearly_mix.filter("year = 2024"))

year,energy_source,category,gwh
2024,wind_onshore,renewable,112548.6
2024,braunkohle,conventional,70978.3
2024,photovoltaik,renewable,63148.4
2024,erdgas,conventional,56913.7
2024,biomasse,renewable,36193.2
2024,steinkohle,conventional,27335.1
2024,wind_offshore,renewable,25665.3
2024,wasserkraft,renewable,17550.3
2024,sonstige_konventionelle,conventional,15839.1
2024,sonstige_erneuerbare,renewable,1058.6


Hourly profile of solar and wind (cell 4)

Question answered: at what time of day do solar and wind produce the most?

In [0]:
%python
profile = (gen.filter(F.col("energy_source").isin("photovoltaik", "wind_onshore", "wind_offshore"))
    .groupBy("hour", "energy_source")
    .agg(F.round(F.avg("mwh"), 1).alias("avg_mwh"))
    .orderBy("hour", "energy_source"))

profile.write.mode("overwrite").saveAsTable("workspace.energy.gold_hourly_profile")
display(profile.filter("energy_source = 'photovoltaik'"))

hour,energy_source,avg_mwh
0,photovoltaik,6.5
1,photovoltaik,5.8
2,photovoltaik,5.6
3,photovoltaik,5.5
4,photovoltaik,6.3
5,photovoltaik,77.6
6,photovoltaik,710.7
7,photovoltaik,2922.5
8,photovoltaik,7347.8
9,photovoltaik,12923.2


Daily generation by category (cell 5)

Question answered: how do renewable and conventional generation develop day by day?

In [0]:
%python
daily = (gen.groupBy("date", "category")
    .agg(F.round(F.sum("mwh"), 1).alias("mwh"))
    .orderBy("date", "category"))

daily.write.mode("overwrite").saveAsTable("workspace.energy.gold_daily_generation")
display(daily.orderBy(F.desc("date")).limit(10))

date,category,mwh
2026-10-22,renewable,1097650.3
2026-10-22,conventional,462700.9
2026-10-21,renewable,571081.8
2026-10-21,conventional,632084.5
2026-10-20,conventional,597322.6
2026-10-20,renewable,786968.0
2026-10-19,renewable,859924.2
2026-10-19,conventional,525358.8
2026-10-18,renewable,718969.2
2026-10-18,conventional,465502.4


Verify all four tables (cell 6)

In [0]:
%python
for t in ["gold_monthly_renewable_share", "gold_yearly_generation_mix",
          "gold_hourly_profile", "gold_daily_generation"]:
    print(t, spark.table(f"workspace.energy.{t}").count(), "rows")

gold_monthly_renewable_share 73 rows
gold_yearly_generation_mix 77 rows
gold_hourly_profile 72 rows
gold_daily_generation 4426 rows
